##### Github repo for module used: https://github.com/m-kovalsky/Fabric/tree/main

In [ ]:
# Install the semantic-link package
%pip install semantic-link

# Import necessary libraries
import sempy.fabric as fabric
import pandas as pd
import datetime as dt
import warnings
from pyspark.sql import SparkSession
from datetime import datetime, timedelta, timezone
warnings.filterwarnings("ignore", category=FutureWarning)


class JiraTimeTracking:
    def __init__(self, workspace_id, dataset_name, update_tables, ignored_tables):
        """
        Initialize the JiraTimeTracking class with workspace and dataset details.
        
        Parameters:
        - workspace_id (str): The ID of the workspace.
        - dataset_name (str): The name of the dataset.
        - update_tables (list): List of tables to update.
        - ignored_tables (list): List of tables to ignore.
        """
        self.workspace_id = workspace_id
        self.dataset_name = dataset_name
        self.update_tables = update_tables
        self.ignored_tables = ignored_tables
        self.tom_server = fabric.create_tom_server(readonly=True, workspace=self.workspace_id)
        self.model = self.tom_server.Databases.GetByName(self.dataset_name).Model
        self.spark = SparkSession.builder.getOrCreate()
    
    def get_last_refresh(self):
        """
        Retrieve the latest refresh datetime of the dataset.
        
        Returns:
        - datetime: latest refresh datetime.
        """
        datasetName = 'JiraTimeTracking - IncrementalRefresh'
        tableName = 'Worklogs'
        rowLimit = 10000
        worklogs_df = fabric.read_table(datasetName, tableName, False, rowLimit)
        # Sort by WorklogCreatedDate in descending order
        sorted_worklogs = worklogs_df.sort_values(by="WorklogCreatedDate", ascending=False)
        # Get the most recent WorklogCreatedDate
        return pd.DataFrame(sorted_worklogs)["WorklogCreatedDate"].iloc[0]

    def get_table_names(self):
        """
        Retrieve the names of tables in the dataset, excluding certain templates.
        
        Returns:
        - list: List of table names.
        """
        table_names = [table.Name for table in self.model.Tables if not table.Name.startswith('LocalDateTable_') and not table.Name.startswith('DateTableTemplate_')]
        return table_names
    
    def replace_data(self):
        """
        Replace table data by reading tables and writing them to Spark with schema merging.
        """
        all_meta_tables_df = pd.DataFrame({'TableName': self.get_table_names()})
        
        for table_name in all_meta_tables_df['TableName']:
            data = fabric.read_table(self.dataset_name, table_name, False)
            df = pd.DataFrame(data)
            df.columns = df.columns.str.replace(' ', '_')
            
            if table_name not in self.update_tables and table_name not in self.ignored_tables:
                spark_df = self.spark.createDataFrame(df)
                spark_df.write.option("mergeSchema", "true").mode("overwrite").format("delta").saveAsTable(table_name)
                print('Table Overwritten: ', table_name)
    
    def get_compare_date(self, compare_date_raw):
        """
        Extract and format the compare date from raw data.
        
        Parameters:
        - compare_date_raw (list): Raw compare date data.
        
        Returns:
        - datetime.date: Formatted compare date.
        """
        compare_date_col = pd.DataFrame(compare_date_raw)['CompareDate']
        compare_date_val = compare_date_col[0]
        year, month, day = map(int, compare_date_val.split("-"))
        return dt.date(year, month, day)
    
    def get_ids_from_fabric(self, compare_date):
        """
        Retrieve distinct issue and worklog IDs from the fabric database based on the compare date.
        
        Parameters:
        - compare_date (datetime.date): The date to compare against.
        
        Returns:
        - DataFrame: DataFrame containing distinct issue and worklog IDs.
        """
        formatted_query = f"""
        SELECT DISTINCT issues.IssueId, worklogs.WorklogId
        FROM issues
        LEFT JOIN worklogs ON issues.IssueId = worklogs.IssueId
        WHERE issues.IssueCreatedDate >= '{compare_date}' OR issues.IssueUpdatedDate >= '{compare_date}'
        """
        ids_fabric_df = self.spark.sql(formatted_query).toPandas()
        return ids_fabric_df

    def get_issue_ids_table(self):
        """
        Retrieve the current issue IDs from the fabric database.
        
        Returns:
        - set: Set of current issue IDs.
        """
        issue_ids_table_df = pd.DataFrame(fabric.read_table(self.dataset_name, 'IssueIds', False))
        return set(issue_ids_table_df['IssueId'].dropna().tolist())

    def get_delete_ids(self):
        """
        Identify issue IDs to delete based on comparison with current IDs.
        
        Returns:
        - DataFrame: DataFrame containing issue IDs to delete.
        """
        fabric_ids_df = self.get_ids_from_fabric('2023-01-01')
        current_ids = self.get_issue_ids_table()
        issues_to_delete = fabric_ids_df[~fabric_ids_df['IssueId'].isin(current_ids)]
        return issues_to_delete

    def calculate_differences(self, issues_current_df, worklogs_current_df, ids_fabric_df):
        """
        Calculate differences between current and fabric issue/worklog IDs.
        
        Parameters:
        - issues_current_df (DataFrame): DataFrame of current issues.
        - worklogs_current_df (DataFrame): DataFrame of current worklogs.
        - ids_fabric_df (DataFrame): DataFrame of fabric issue and worklog IDs.
        
        Returns:
        - tuple: Sets of issue/worklog IDs to insert, update, and delete.
        """
        issue_ids_fabric = set(ids_fabric_df['IssueId'].dropna().tolist())
        worklog_ids_fabric = set(ids_fabric_df['WorklogId'].dropna().tolist())

        issue_ids_current = set(issues_current_df['IssueId'])
        worklog_ids_current = set(worklogs_current_df['WorklogId'])

        issue_ids_to_insert = issue_ids_current - issue_ids_fabric
        issue_ids_to_update = issue_ids_fabric & issue_ids_current - issue_ids_to_insert

        worklog_ids_to_insert = worklog_ids_current - worklog_ids_fabric
        worklog_ids_to_update = worklog_ids_fabric & worklog_ids_current - worklog_ids_to_insert
        worklog_ids_to_delete = worklog_ids_fabric - worklog_ids_current

        return (issue_ids_to_insert, issue_ids_to_update, worklog_ids_to_insert, worklog_ids_to_update, worklog_ids_to_delete)


# Set variables
workspace_id = '9300f60b-4d8e-4b03-b215-9a6e44c0cec7'
dataset_name = 'JiraTimeTracking - IncrementalRefresh'
update_tables = ['Issues', 'Worklogs']
ignored_tables = ['MeasuresTable']

# Initialte Class 
jtt_class = JiraTimeTracking(workspace_id, dataset_name, update_tables, ignored_tables)
latest_refresh_end = jtt_class.get_last_refresh()


# Check if the latest refresh was within the past 24 hours
if latest_refresh_end:
    now = datetime.now(timezone.utc) 
    latest_refresh_end = latest_refresh_end.replace(tzinfo=timezone.utc)
    if now - latest_refresh_end <= timedelta(hours=24):
        print(f'Refresh Detected; Merge Script Running Now.')

        jtt_class.replace_data()

        # Read raw data from the 'Issues', 'Worklogs', and 'CompareDate' tables
        issues_raw = fabric.read_table(dataset_name, 'Issues', False)
        worklogs_raw = fabric.read_table(dataset_name, 'Worklogs', False)
        compare_date_raw = fabric.read_table(dataset_name, 'CompareDate', False)

        # Get the comparison date from the raw compare date data
        compare_date = jtt_class.get_compare_date(compare_date_raw)

        # Convert raw data into pandas DataFrames for easier manipulation
        issues_current_df = pd.DataFrame(issues_raw)
        worklogs_current_df = pd.DataFrame(worklogs_raw)

        # Retrieve IDs from the fabric based on the comparison date
        ids_fabric_df = jtt_class.get_ids_from_fabric(compare_date)

        # Calculate the differences between the current data and the fabric data
        (issue_ids_to_insert, issue_ids_to_update, worklog_ids_to_insert, worklog_ids_to_update, worklog_ids_to_delete) = jtt_class.calculate_differences(issues_current_df, worklogs_current_df, ids_fabric_df)

        # Merge operations for issues

        # Filter issues that need to be inserted or updated
        issues_to_merge = issues_current_df[issues_current_df['IssueId'].isin(issue_ids_to_insert | issue_ids_to_update)]
        # Convert the filtered issues DataFrame to a Spark DataFrame
        issues_to_merge_spark_df = jtt_class.spark.createDataFrame(issues_to_merge)
        # Create or replace a temporary view for the issues to be merged
        issues_to_merge_spark_df.createOrReplaceTempView("issues_to_merge_temp")

        # Define the SQL merge query for issues
        merge_query = """
        MERGE INTO issues AS target
        USING issues_to_merge_temp AS source
        ON target.IssueId = source.IssueId
        WHEN MATCHED THEN
        UPDATE SET *
        WHEN NOT MATCHED THEN
        INSERT * 
        """
        # Execute the merge query
        jtt_class.spark.sql(merge_query)

        # Merge operations for worklogs

        # Filter worklogs that need to be inserted or updated
        worklogs_to_merge = worklogs_current_df[worklogs_current_df['WorklogId'].isin(worklog_ids_to_insert | worklog_ids_to_update)]
        # Convert the filtered worklogs DataFrame to a Spark DataFrame
        worklogs_to_merge_spark_df = jtt_class.spark.createDataFrame(worklogs_to_merge)
        # Create or replace a temporary view for the worklogs to be merged
        worklogs_to_merge_spark_df.createOrReplaceTempView("worklogs_to_merge_temp")

        # Define the SQL merge query for worklogs
        merge_query = """
        MERGE INTO worklogs AS target
        USING worklogs_to_merge_temp AS source
        ON target.WorklogId = source.WorklogId
        WHEN MATCHED THEN
        UPDATE SET *
        WHEN NOT MATCHED THEN
        INSERT *
        """
        # Execute the merge query
        jtt_class.spark.sql(merge_query)

        # If there are worklog IDs to delete, format and execute the delete query
        if worklog_ids_to_delete:
            formatted_query = f"DELETE FROM worklogs WHERE WorklogId IN ({','.join(map(str, worklog_ids_to_delete))})"
            jtt_class.spark.sql(formatted_query)

        # Extract distinct IssueIds and WorklogIds from fabric_ids_tbd and delete them
        fabric_ids_tbd = jtt_class.get_delete_ids()
        distinct_issue_ids = set(fabric_ids_tbd['IssueId'].dropna().tolist())
        distinct_worklog_ids = set(fabric_ids_tbd['WorklogId'].dropna().tolist())

        # If there are distinct issue IDs to delete, format and execute the delete query
        if distinct_issue_ids:
            formatted_query = f"DELETE FROM issues WHERE IssueId IN ({','.join(map(str, distinct_issue_ids))})"
            jtt_class.spark.sql(formatted_query)

        # If there are distinct worklog IDs to delete, format and execute the delete query
        if distinct_worklog_ids:
            formatted_query = f"DELETE FROM worklogs WHERE WorklogId IN ({','.join(map(str, distinct_worklog_ids))})"
            jtt_class.spark.sql(formatted_query)

        print("Merge operations completed successfully.")
else:
    print(f'No refresh data found for dataset {dataset_name}')